# Deliverable 2 - INF201 



## Task 1: File ingestion and regular expressions



In [1]:
from pathlib import Path

def read_text_file(file_path: str | Path) -> tuple[str, str]: 
    """Leser filen ved bruk av UTF-8 eller ISO-8859-1 encoding"""
    
    file_path = Path(file_path)

    try:
        text = file_path.read_text(encoding="utf-8")          # Prøver først å lese filen med UTF-8 encoding
        encoding_used = "utf-8"
    except UnicodeDecodeError:
        text = file_path.read_text(encoding="ISO-8859-1")     # Hvis UTF-8 feiler, prøver vi med ISO-8859-1
        encoding_used = "ISO-8859-1"

    return text, encoding_used


In [2]:
import re

def parse_log(text: str) -> list[dict]: 
    """Finner gyldige transaksjoner i loggfilen og returnerer dem som en liste av ordbøker"""

    pattern = re.compile(
        r"\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\] " # Henter dato og klokkeslett for transaksjonen
        r"\[(?P<register_id>\w+)\] "                               # Henter register-ID
        r"item=(?P<item>\w+) "                                     # Henter varenavn
        r"price=(?P<price>\d+\.\d{2}) "                            # Henter prisen på varen
        r"(?P<currency>EUR|NOK|USD) "                              # Henter valutaen
        r"\((?P<status>SUCCESS|REFUND|CANCELLED)\)"                # Henter statusen for transaksjonen
    )

    transactions = []

    for match in pattern.finditer(text):                           # Går gjennom teksten og finner alle steder som matcher regex-mønsteret
        transaction = match.groupdict()                            # Lager en ordbok med transaksjonsdataene
        transaction["price"] = float(transaction["price"])         # Konverterer prisen om fra tekst til tall.
        transactions.append(transaction)

    return transactions


# Test
test_file = Path("data/2026/library_cafe/POS_01/register_sales.log")

text, encoding = read_text_file(test_file)
transactions = parse_log(text)

print("Encoding:", encoding)
print("Antall transaksjoner:", len(transactions))
print(transactions[:2])                                  # Printer de to første transaksjonene for å sjekke at alt fungerer som det skal

Encoding: utf-8
Antall transaksjoner: 80
[{'timestamp': '2026-10-15 07:00:41', 'register_id': 'POS_01', 'item': 'Cappuccino', 'price': 3.82, 'currency': 'NOK', 'status': 'SUCCESS'}, {'timestamp': '2026-10-15 07:03:07', 'register_id': 'POS_01', 'item': 'Croissant', 'price': 3.99, 'currency': 'NOK', 'status': 'SUCCESS'}]


I task 1 ble loggfilene lest ved først å prøve UTF-8 og deretter ISO-8859-1 dersom UTF-8 ikke fungerte. Regex ble brukt til å hente ut de relevante feltene fra gyldige transaksjonslinjer, mens støylinjer ble ignorert. Denne løsningen fungerer godt fordi loggfilene følger et tydelig mønster. Samtidig er regexen avhengig av at formatet på logglinjene ikke endres vesentlig.

## Task 2: Recursive utility functions



In [3]:
def find_log_files(folder: str | Path) -> list[Path]:     
    """Finner alle loggfiler i en gitt mappe og returnerer dem som en liste av Path-objekter"""

    folder = Path(folder)
    log_files = []

    for item in folder.iterdir(): 
        if item.is_dir():
            log_files.extend(find_log_files(item))        # Rekursivt søker gjennom undermapper for loggfiler

        elif item.is_file() and item.suffix == ".log":    # Sjekker om filen har .log som filendelse, og legger den til i listen over loggfiler
            log_files.append(item)
    
    return log_files

# Test
log_files = find_log_files("data")

print("Antall loggfiler:", len(log_files))
print(log_files[:3])                                      # Printer de tre første loggfilene for å sjekke at alt fungerer som det skal

Antall loggfiler: 9
[PosixPath('data/2026/science_building/POS_01/register_sales.log'), PosixPath('data/2026/science_building/POS_03/register_sales.log'), PosixPath('data/2026/science_building/POS_02/register_sales.log')]


In [4]:
import json

def find_active_registers(data) -> set[str]:                         
    """Finner ID-ene til alle aktive registre i en nestet datastruktur"""
    
    active_registers = set()                                         # Vil ha unike register-IDer.

    if isinstance(data, dict):
        if data.get("active") is True and "id" in data:              # Legger til register ID dersom registeret er aktivt.
            active_registers.add(data["id"])
            
        for value in data.values():                                  # Går gjennom alle verdier i ordboken og søker etter aktive registre i dem.
            active_registers.update(find_active_registers(value)) 

    elif isinstance(data, list):                                     # Går gjennom alle elementer i listen og søker etter aktive registre i dem.
        for item in data:
            active_registers.update(find_active_registers(item))
        
    return active_registers

# Test
config_file = Path("data/store_config.json")

with open(config_file, "r", encoding="utf-8") as file:
    config_data = json.load(file)

active_registers = find_active_registers(config_data)

print("Aktive registre:", active_registers)

Aktive registre: {'POS_03', 'POS_01', 'POS_02'}


Rekursjon brukt både for å søke gjennom mappestrukturen og for å hente ut aktive registre fra den nestede JSON-strukturen. Dette passer godt fordi både mapper og JSON-data kan inneholde flere nivåer. Rekursjon gjør løsningen fleksibel, men det kan også være vanskeligere å følge programflyten sammenlignet med en vanlig løkke.

## Task 3: Functions for cleaning and analysing data using NumPy


In [5]:
import numpy as np

def clean_price_outliers(prices) -> np.ndarray:     
    """Fjerner ekstreme prisavvik og erstatter dem med medianen."""

    prices = np.array(prices, dtype=float)          # Konverterer listen over priser til en numpy array for enklere beregninger.

    mean = np.mean(prices)                          # Beregner gjennomsnittet av prisene.
    std = np.std(prices)                            # Beregner standardavviket av prisene.

    outliers = np.abs(prices - mean) > 3 * std      # Identifiserer priser som er mer enn 3 standardavvik fra gjennomsnittet.

    median = np.median(prices[~outliers])           # Beregner medianen av prisene som ikke er outliers.

    prices[outliers] = median                       # Erstatter outliers med medianen.

    return prices

# Testcelle
test_prices = [
    3.00, 4.50, 4.20, 3.50, 3.80,
    7.50, 3.20, 4.70, 4.10, 3.60,
    3.90, 7.20, 3.10, 4.40, 4.30,
    999.00
]

cleaned_prices = clean_price_outliers(test_prices)

print("Før:",test_prices)
print("Etter:", cleaned_prices)

Før: [3.0, 4.5, 4.2, 3.5, 3.8, 7.5, 3.2, 4.7, 4.1, 3.6, 3.9, 7.2, 3.1, 4.4, 4.3, 999.0]
Etter: [3.  4.5 4.2 3.5 3.8 7.5 3.2 4.7 4.1 3.6 3.9 7.2 3.1 4.4 4.3 4.1]


In [6]:
def compute_refund_ratio(
        sales_count: np.ndarray,
        refund_count: np.ndarray
) -> np.ndarray:
    """Beregner refusjonsforholdet elementvis for hvert salgstall"""

    return refund_count / (sales_count + 1e-6)  # Legger til en liten verdi for å unngå deling på null

NumPy ble brukt til å finne ekstreme prisavvik og erstatte dem med medianen av de gyldige prisene. Dette hindrer at feilregistreringer som 999.00 påvirker den videre analysen. I testen ble 999 byttet ut med 4.1. Metoden fungerer godt her, fordi generatorskriptet gir hver transaksjon bare 3 % sjanse for å få en feilpris. Siden metoden bruker gjennomsnitt og standardavvik, drar feilverdiene likevel disse tallene opp. Hvis mer enn ca. 10 % av prisene var feil, ville feilene ikke lenger skilt seg nok ut til å bli fanget.

## Task 4: Data analysis using Pandas


In [7]:
# Bygge en DataFrame
import pandas as pd

def build_dataframe(transactions: list[dict]) -> pd.DataFrame:
    """Lager en sortert DataFrame fra en liste med transaksjoner"""
    
    df = pd.DataFrame(transactions)                    

    df["timestamp"] = pd.to_datetime(df["timestamp"])  # Konverterer timestamp om til datetime

    df = df.set_index("timestamp")                     # Bruker timestamp kolonnen som indeks for DataFrame
    df = df.sort_index()                               # Sorterer DataFrame etter timestamp

    return df

# Test
df = build_dataframe(transactions)
print(df.head())

                    register_id        item  price currency     status
timestamp                                                             
2026-10-15 07:00:41      POS_01  Cappuccino   3.82      NOK    SUCCESS
2026-10-15 07:03:07      POS_01   Croissant   3.99      NOK    SUCCESS
2026-10-15 07:03:53      POS_01       Latte   4.90      USD    SUCCESS
2026-10-15 07:06:48      POS_01    Espresso   2.93      EUR     REFUND
2026-10-15 07:08:41      POS_01       Latte   4.13      EUR  CANCELLED


In [8]:
# Inntekt per time
def hourly_revenue(df: pd.DataFrame) -> pd.DataFrame:
    """Beregner totalpris per vare i intervaller på en time"""

    result = (
        df.groupby("item")          # Grupperer DataFrame etter varenavn
        .resample("1h")["price"]    # Deler tidsdataene inn i 1-times intervaller og velger priskolonnen
        .sum()
        .reset_index()
    )

    return result

# Test
hourly = hourly_revenue(df)
print(hourly.head())

         item           timestamp    price
0  Cappuccino 2026-10-15 07:00:00  1035.59
1  Cappuccino 2026-10-15 08:00:00  1015.38
2  Cappuccino 2026-10-15 09:00:00     7.67
3   Croissant 2026-10-15 07:00:00    29.91
4   Croissant 2026-10-15 08:00:00    20.44


In [9]:
# Pivot-tabell
def create_status_pivot(df: pd.DataFrame) -> pd.DataFrame: 
    """Lager en pivottabell med total pris per vare og status"""

    pivot = pd.pivot_table(
        df,
        values="price",                # Verdien vi ønsker å summere
        index="item",                  # En rad per vare
        columns="status",              # En kolonne per status
        aggfunc="sum",                 # Summerer prisene
        fill_value=0                   # Fyller tomme celler med 0
    )

    return pivot

# Test
status_pivot = create_status_pivot(df)
print(status_pivot)

status      CANCELLED   REFUND  SUCCESS
item                                   
Cappuccino      11.94     0.00  2046.70
Croissant       24.30     3.60    37.64
Espresso         0.00    15.51    20.18
Latte            8.79  1012.87    22.83
Muffin           7.82     3.48    19.94
Sandwich         7.94    22.74  1067.06


In [10]:
# Regne ut refusjonsforhold
def item_refund_ratios(df: pd.DataFrame) -> pd.DataFrame:
    """Beregner refusjonsforholdet for hver vare"""

    counts = (
        df.groupby(["item", "status"])                           # Teller antall transaksjoner per vare og status
        .size()
        .unstack(fill_value=0)
    )

    sales_count = counts ["SUCCESS"].to_numpy()
    refund_count = counts ["REFUND"].to_numpy()

    ratios = compute_refund_ratio(sales_count, refund_count)

    result = pd.DataFrame({
        "item": counts.index,
        "refund_ratio": ratios
    })

    return result

# Test
refund_ratios = item_refund_ratios(df)
print(refund_ratios)

         item  refund_ratio
0  Cappuccino      0.000000
1   Croissant      0.100000
2    Espresso      0.714286
3       Latte      0.800000
4      Muffin      0.166667
5    Sandwich      0.300000


Transaksjonene ble samlet i en Pandas DataFrame med tidspunkt som indeks. Dette gjorde det enkelt å gruppere data etter vare, status og tidsintervall, og å lage både timebaserte summer og pivottabeller. Testene i denne delen er kjørt på en fil før rensing, så de høye tallene, for eksempel 1035 for Cappuccino, kommer fra 999-feilene som fjernes i task 5. En begrensning er at summene blander EUR, NOK og USD uten å regne dem om, så de er ikke ekte inntekt i en valuta.

## Task 5: Create full workflow


In [11]:
log_files = find_log_files("data")                     # Finner alle loggfiler i data-mappen og dens undermapper

config_file = Path("data/store_config.json")           # Leser konfigurasjonsfilen og finner aktive registre

with open(config_file, "r", encoding="utf-8") as file:
    config_data = json.load(file)

active_registers = find_active_registers(config_data)


all_transactions = []

for log_file in log_files:
    text, encoding = read_text_file(log_file)          # Leser innholdet i loggfilen
    transactions = parse_log(text)                     # Parser loggfilen for transaksjoner
    all_transactions.extend(transactions)              # Legger til transaksjonene i listen over alle transaksjoner

# Lager en DataFrame fra listen av alle transaksjoner
df_all = build_dataframe(all_transactions)                     

df_all = df_all[df_all["register_id"].isin(active_registers)]  # Filtrerer DataFrame for å kun inkludere transaksjoner fra aktive registre

# Fjerner ekstreme prisavvik
df_all["price"] = clean_price_outliers(df_all["price"])  

#  Korrigerer NOK-priser som feilaktig er registrert i EUR-skala
df_all.loc[df_all["currency"] == "NOK", "price"] *= 10   

# Lager rapportene
hourly_report = hourly_revenue(df_all)      
status_pivot = create_status_pivot(df_all)  
refund_ratios = item_refund_ratios(df_all)  

# Lagrer resultatene til filer
df_all.to_csv(
    "cleaned_transactions.csv",
    index_label="timestamp"
)

hourly_report.to_json(
    "hourly_revenue.json",
    orient="records",
    date_format="iso",
    indent=2
)

status_pivot.to_csv(
    "status_pivot_table.csv"
)

refund_ratios.to_csv(
    "item_refund_ratios.csv",
    index=False
)

# Test
print("Antall loggfiler:", len(log_files))
print("Antall transaksjoner:", len(df_all))
print("Høyeste pris etter rengjøring:", df_all["price"].max())
print("Aktive registre:", active_registers)

Antall loggfiler: 9
Antall transaksjoner: 720
Høyeste pris etter rengjøring: 79.9
Aktive registre: {'POS_03', 'POS_01', 'POS_02'}


Til slutt ble alle funksjonene satt sammen i en samlet arbeidsflyt. Transaksjonene ble filtrert på aktive registre, men filteret fjernet ingen, fordi den eneste inaktive kassen (POS_04) ikke har loggfiler. Deretter ble prisavvik renset, NOK-priser korrigert og resultatene lagret som CSV- og JSON-filer. Prisavvikene fjernes før NOK-prisene ganges med 10, ellers ville de høyere NOK-prisene påvirket hva som regnes som et avvik. Vi fikk 720 transaksjoner (9 filer × 80), og den høyeste prisen etter rensing er 79.9, så 999-feilene er borte. Denne delen viser også fordelen med å dele løsningen opp i små funksjoner som kan testes hver for seg og deretter brukes sammen.